# Additive + multi-layer steering

Two experiments sharing the same scaffold.

- **4a. Multi-layer single direction**: inject the layer-14 pivotal vector at {14,16} and {14,16,20} simultaneously via additive hooks. Compare against factor=1.2 and factor=1.4 applied at layer 14 alone (matched injected energy).
- **4b. Multi-direction single layer**: at layer 14, inject 0.2*v14 + 0.2*v16 (both 1024-d). Tests whether different probe directions combine constructively.


## 1. Install

In [ ]:
# Run once per runtime. Safe to skip if already installed.
%pip install -q "transformers>=4.44" datasets accelerate matplotlib seaborn scikit-learn tqdm pandas


## 2. Imports + seed

In [ ]:
import json
import os
import random
import re
import sys
import time
import zipfile
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.bfloat16
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = "mps"
    DTYPE = torch.float32
else:
    DEVICE = "cpu"
    DTYPE = torch.float32

RUN_TAG = "additive_multilayer"
RESULTS_DIR = Path("nb_results") / RUN_TAG
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"device={DEVICE} dtype={DTYPE} run_tag={RUN_TAG}")
print(f"results -> {RESULTS_DIR.resolve()}")


## 3. Config

In [ ]:
MAX_EXAMPLES = 100
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.6
TOP_P = 0.9
FORCE_RERUN = False
ALPHA = 0.2  # per-layer injected strength for multi-layer variants


## 4. Probes (layers 14, 16, 20)

In [ ]:
# Probe location.  Fallback order: local repo -> GitHub raw -> manual upload.
LOCAL_PROBE_DIR = Path("artifacts/cached3/sklearn/steering_configs")
PROBE_RAW_URL_BASE = (
    "https://raw.githubusercontent.com/"
    "stvngo/Pivotal-Token-Representation-Learning/"
    "main/artifacts/cached3/sklearn/steering_configs/"
)


In [ ]:
import urllib.request


def load_probe(layer: int, local_dir: Path = LOCAL_PROBE_DIR):
    cfg_name = f"steering_layer{layer}.json"
    vec_name = f"steering_layer{layer}_vector.npy"
    cfg_path = local_dir / cfg_name
    vec_path = local_dir / vec_name
    if not (cfg_path.exists() and vec_path.exists()):
        dest = RESULTS_DIR / "probes"
        dest.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(PROBE_RAW_URL_BASE + cfg_name, dest / cfg_name)
            urllib.request.urlretrieve(PROBE_RAW_URL_BASE + vec_name, dest / vec_name)
            cfg_path, vec_path = dest / cfg_name, dest / vec_name
        except Exception as exc:
            raise RuntimeError(
                f"Could not locate steering probe for layer {layer}. "
                f"Either clone the repo, fix PROBE_RAW_URL_BASE, or upload the files. ({exc})"
            ) from exc
    cfg = json.loads(cfg_path.read_text())
    vec = np.load(vec_path).astype(np.float32)
    return cfg, vec, cfg_path, vec_path


In [ ]:
# --- Google Colab: upload your own probe --------------------------------
# Uncomment if the automatic fallback above fails.
#
# from google.colab import files
# uploaded = files.upload()
# print('Uploaded:', list(uploaded.keys()))


In [ ]:
probes = {}
for L in (14, 16, 20):
    cfg, vec, _, _ = load_probe(L)
    probes[L] = {"cfg": cfg, "vector": vec, "tensor": torch.from_numpy(vec).to(torch.float32)}
    print(f"layer {L}: norm={cfg['vector_norm']:.4f}")
v14 = probes[14]["tensor"]; v16 = probes[16]["tensor"]; v20 = probes[20]["tensor"]


## 5. Model + hooks + parser + data

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-0.6B"

_t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
).to(DEVICE)
model.eval()
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
print(
    f"Loaded {MODEL_NAME} on {DEVICE} ({DTYPE}) "
    f"in {time.time() - _t0:.1f}s | hidden_size={model.config.hidden_size} "
    f"layers={model.config.num_hidden_layers}"
)


In [ ]:
import torch.nn as nn


def _get_decoder_layer(mdl: nn.Module, layer_idx: int) -> nn.Module:
    if hasattr(mdl, "model") and hasattr(mdl.model, "layers"):
        return mdl.model.layers[layer_idx]
    if hasattr(mdl, "transformer") and hasattr(mdl.transformer, "h"):
        return mdl.transformer.h[layer_idx]
    if hasattr(mdl, "decoder") and hasattr(mdl.decoder, "layers"):
        return mdl.decoder.layers[layer_idx]
    raise AttributeError(f"Cannot find decoder layers in {type(mdl)}")


def _make_steering_hook(vector: torch.Tensor, strength: float):
    def hook(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        v = vector.to(hidden.device).to(hidden.dtype)
        if v.dim() == 1:
            v = v.view(1, 1, -1)
        delta = strength * v
        if isinstance(output, tuple):
            return (hidden + delta,) + output[1:]
        return hidden + delta
    return hook


def register_steering(mdl, layer: int, vector: torch.Tensor, strength: float):
    return [
        _get_decoder_layer(mdl, layer).register_forward_hook(
            _make_steering_hook(vector, strength)
        )
    ]


def register_additive_hooks(mdl, injections):
    '''Register one additive hook per (layer_idx, vector, strength) tuple.'''
    handles = []
    for layer_idx, vector, strength in injections:
        handles.append(
            _get_decoder_layer(mdl, layer_idx).register_forward_hook(
                _make_steering_hook(vector, strength)
            )
        )
    return handles


def remove_hooks(handles):
    for h in handles:
        h.remove()


In [ ]:
def extract_gsm8k_answer(text: str):
    m = re.search(r"####\s*([+-]?\d+(?:,\d{3})*(?:\.\d+)?)", text)
    if m:
        return m.group(1).replace(",", "").strip()
    nums = re.findall(r"[-+]?\d*\.?\d+", text)
    return nums[-1] if nums else None


def is_correct(pred, gt):
    if pred is None or gt is None:
        return False
    p = pred.strip().replace(",", "")
    g = gt.strip().replace(",", "")
    try:
        return float(p) == float(g)
    except ValueError:
        return p == g


def compute_metrics(results):
    n = len(results)
    if n == 0:
        return {"accuracy": 0.0, "f1": 0.0, "parse_rate": 0.0, "correct": 0, "n": 0}
    correct = sum(1 for r in results if r["correct"])
    parsed = sum(1 for r in results if r["predicted"] is not None)
    tp = correct
    fp = parsed - correct
    fn = n - correct
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {
        "accuracy": correct / n,
        "f1": f1,
        "parse_rate": parsed / n,
        "correct": correct,
        "n": n,
    }


# Inline parser sanity checks (mirror tests/test_answer_extraction.py).
assert extract_gsm8k_answer("#### 42") == "42"
assert extract_gsm8k_answer("He has #### 1,234 apples.") == "1234"
assert extract_gsm8k_answer("The net change is #### -7") == "-7"
assert extract_gsm8k_answer("So the answer is #### 3.50") == "3.50"
assert extract_gsm8k_answer("...so the total is 18 apples.") == "18"
assert extract_gsm8k_answer("") is None
assert is_correct("3.0", "3") is True
assert is_correct("1234", "1,234") is True
assert is_correct(None, "5") is False
print("Parser sanity checks passed.")


In [ ]:
from datasets import load_dataset

_ds_full = load_dataset("openai/gsm8k", "main", split="test")
_rng = np.random.default_rng(SEED)
_indices = sorted(
    _rng.choice(len(_ds_full), size=min(MAX_EXAMPLES, len(_ds_full)), replace=False).tolist()
)
ds_subset = _ds_full.select(_indices)
examples = [
    {"question": row["question"],
     "ground_truth": extract_gsm8k_answer(row["answer"]),
     "answer_full": row["answer"]}
    for row in ds_subset
]
print(f"GSM8K test subset: {len(examples)} examples (seed={SEED})")


In [ ]:
def generate_once(prompt: str) -> str:
    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=True).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)


def _reseed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)


def run_eval_with_hooks(label, register_fn, desc_prefix=""):
    '''register_fn(model) -> list of hook handles (or []). Runs paired GSM8K subset.'''
    _reseed()
    handles = register_fn(model) if register_fn is not None else []
    results = []
    try:
        for i, ex in enumerate(tqdm(examples, desc=f"{desc_prefix}{label}")):
            prompt = f"Question: {ex['question']}\n\nLet's think step by step.\n\n"
            text = generate_once(prompt)
            pred = extract_gsm8k_answer(text)
            ok = is_correct(pred, ex["ground_truth"])
            results.append({
                "idx": i,
                "question": ex["question"],
                "ground_truth": ex["ground_truth"],
                "predicted": pred,
                "correct": ok,
                "full_output": text[-1000:],
            })
    finally:
        remove_hooks(handles)
    return results, compute_metrics(results)


def save_run(label, results, metrics, extra_state=None):
    (RESULTS_DIR / f"{label}_results.json").write_text(json.dumps(results, indent=2))
    (RESULTS_DIR / f"{label}_generations.txt").write_text(
        "\n\n====\n\n".join(
            f"[{r['idx']}] gt={r['ground_truth']} pred={r['predicted']} correct={r['correct']}\n{r['full_output']}"
            for r in results
        )
    )
    state = {
        "label": label,
        "model": MODEL_NAME,
        "seed": SEED,
        "max_examples": len(examples),
        "max_new_tokens": MAX_NEW_TOKENS,
        "device": DEVICE,
        "run_tag": RUN_TAG,
        "metrics": metrics,
    }
    if extra_state:
        state.update(extra_state)
    (RESULTS_DIR / f"{label}_state.json").write_text(json.dumps(state, indent=2))
    return state


## 6. Base

In [ ]:
base_cache = RESULTS_DIR / "base_results.json"
if base_cache.exists() and not FORCE_RERUN:
    base_results = json.loads(base_cache.read_text())
    base_metrics = compute_metrics(base_results)
else:
    base_results, base_metrics = run_eval_with_hooks("base", register_fn=None)
    save_run("base", base_results, base_metrics)
print(f"base: acc={base_metrics['accuracy']:.3f}")


## 7. Variant specifications

In [ ]:
def inj_single(layer, vec, strength):
    return [(layer, vec, strength)]

variants = {
    # Multi-layer, single direction (v14 applied at multiple layers).
    "L14_f1.2":          inj_single(14, v14, 0.2),
    "L14_f1.4":          inj_single(14, v14, 0.4),
    "L14+16_alpha0.2":   inj_single(14, v14, 0.2) + inj_single(16, v14, 0.2),
    "L14+16+20_alpha0.2": inj_single(14, v14, 0.2) + inj_single(16, v14, 0.2) + inj_single(20, v14, 0.2),
    # Multi-direction, single layer.
    "L14_v14+v16":       [(14, v14, 0.2), (14, v16, 0.2)],
    "L14_v14+v20":       [(14, v14, 0.2), (14, v20, 0.2)],
}

def injected_energy(spec):
    return float(np.sum([abs(s) * v.norm().item() for _, v, s in spec]))

for k, spec in variants.items():
    print(f"{k}: energy={injected_energy(spec):.3f} ({len(spec)} hooks)")


## 8. Run

In [ ]:
all_metrics = {"base": base_metrics}

for label, spec in variants.items():
    cache = RESULTS_DIR / f"{label}_results.json"
    if cache.exists() and not FORCE_RERUN:
        res = json.loads(cache.read_text())
        met = compute_metrics(res)
        print(f"[cached] {label}: acc={met['accuracy']:.3f}")
    else:
        def reg(mdl, sp=spec):
            return register_additive_hooks(mdl, sp)
        res, met = run_eval_with_hooks(label, register_fn=reg)
        save_run(label, res, met, extra_state={
            "injections": [(L, float(s), float(v.norm().item())) for L, v, s in spec],
            "energy": injected_energy(spec),
        })
        print(f"[done]   {label}: acc={met['accuracy']:.3f}")
    all_metrics[label] = met


## 9. Plots + table

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 110

rows = []
for label in list(all_metrics.keys()):
    m = all_metrics[label]
    energy = injected_energy(variants[label]) if label in variants else 0.0
    rows.append({"variant": label, "energy": energy,
                 "accuracy": m["accuracy"], "f1": m["f1"], "parse_rate": m["parse_rate"]})
df = pd.DataFrame(rows)
df.to_csv(RESULTS_DIR / "summaries.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(df["variant"], df["accuracy"])
ax.axhline(base_metrics["accuracy"], ls="--", color="gray", label="base")
ax.set_ylabel("accuracy"); ax.set_xticklabels(df["variant"], rotation=30, ha="right")
ax.set_title("Additive / multi-layer steering")
ax.legend(); fig.tight_layout()
fig.savefig(RESULTS_DIR / "additive_bars.png"); plt.show()

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.scatter(df["energy"], df["accuracy"])
for _, r in df.iterrows():
    ax.annotate(r["variant"], (r["energy"], r["accuracy"]), fontsize=8, alpha=0.8)
ax.axhline(base_metrics["accuracy"], ls="--", color="gray", label="base")
ax.set_xlabel("injected energy (sum |strength|*|vector|)")
ax.set_ylabel("accuracy")
ax.set_title("Accuracy vs injected energy")
ax.grid(alpha=0.3); ax.legend(); fig.tight_layout()
fig.savefig(RESULTS_DIR / "energy_vs_accuracy.png"); plt.show()
df.sort_values("accuracy", ascending=False)


## 10. Bundle

In [ ]:
zip_path = Path(f"nb_results_{RUN_TAG}.zip")
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in RESULTS_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=p.relative_to(RESULTS_DIR.parent))
print(f"Zipped {zip_path} ({zip_path.stat().st_size / 1024:.1f} KB)")

try:
    from google.colab import files  # type: ignore
    files.download(str(zip_path))
except Exception:
    print("Not on Colab — the zip is on the local filesystem.")
